# 🇮🇳 Fine-tune LFM2.5-350M for Hindi + Hinglish — Complete Beginner Guide

This notebook takes Liquid AI's **LFM2.5-350M-Base** model and teaches it Hindi and
Hinglish (Hindi written in English letters, like "aap kaise ho?") on a **free
Google Colab T4 GPU**.

## What we'll do (overview)
| Step | What | Time on T4 |
|---|---|---|
| 0 | Check GPU & connect Drive | 2 min |
| 1 | Install tools | 5 min |
| 2 | Get the code | 1 min |
| 3 | Download + prepare Hindi/Hinglish data | 10-20 min |
| 4 | **LCPT** — teach the model to read Hindi (raw stories) | 60-90 min |
| 5 | **SFT** — teach it to chat in Hindi/Hinglish | 30-50 min |
| 6 | Test it live | 5 min |
| 7 | (Optional) Export GGUF for Ollama/llama.cpp | 10 min |
| 8 | Save everything to Drive / Hugging Face | 5 min |

## How to use this notebook
- Click the ▶️ button on the left of each cell, **in order, top to bottom**
- Wait for each cell to finish (spinner stops) before running the next
- `!` at the start of a line = a terminal command; lines without it = Python

**Runtime → Change runtime type → T4 GPU** should already be set. Cell 1 checks it.

## Step 0 — Check your GPU

Colab gives you a free T4 GPU (16GB). This cell confirms it's active.
If it prints "NO GPU", go to **Runtime → Change runtime type → T4 GPU → Save**, then re-run.

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ NO GPU! Runtime → Change runtime type → T4 GPU → Save, then re-run this cell.")

## Step 1 — Mount Google Drive

Your Colab files are **wiped when the session ends**. Google Drive keeps your
trained model safe. This opens a permission popup — click your account → Allow.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create our project folder in Drive (where the final model will be saved)
import os
os.makedirs('/content/drive/MyDrive/lfm25_hindi', exist_ok=True)
print("✅ Drive mounted — model will be saved to /content/drive/MyDrive/lfm25_hindi")

## Step 2 — Install the tools

- `transformers`, `peft`, `trl` — the finetuning stack (Hugging Face)
- `datasets` — to download training data
- `sentencepiece` — the tokenizer format LFM2.5 uses

⏱️ Takes ~3-5 minutes. Ignore any "pip dependency resolver" warnings — they're harmless.

In [ ]:
# %%time
%pip install -q -U transformers peft trl datasets accelerate sentencepiece pyyaml
print("✅ Installed")

## Step 3 — Get the project code

Clones your GitHub repo (training scripts + configs). Everything we do next
uses these scripts.

In [ ]:
%cd /content
!git clone https://github.com/SujitRoy/LFM2.5-350M-Base.git
%cd /content/LFM2.5-350M-Base
!ls scripts/ configs/

## Step 4 — Download the datasets (~4 min)

This runs our verified dataset stack (all were manually checked on HF Hub).
Total: **~660,000 Hindi/Hinglish instruction pairs**.


In [ ]:
# %%time
%cd /content/LFM2.5-350M-Base

# Downloads 5 SFT datasets (~637K pairs). Wikipedia is skipped (we use TinyStories instead)
!python scripts/download_data.py --skip_wiki

## Step 5 — Download & parse TinyStories-Hindi (the fluency corpus)

A 2.6 GB parallel corpus (484K English↔Hindi story pairs, machine-translated with
IndicTrans2). From it we extract:
- **LCPT corpus**: 400,000 Hindi stories (raw reading practice for the model)
- **25,000 EN→HI translation pairs** (added to SFT mix)


In [ ]:
# %%time
# ~4 min download
!wget -q -O /content/tiny_hi.txt "https://huggingface.co/datasets/Meyank/Tiny_Stories_Hindi/resolve/main/translations_indictrans2-200m-2.txt"

# Parse into LCPT corpus + translation SFT pairs (~20 s, streams — low RAM)
!python scripts/parse_tinystories_hindi.py \
    --input /content/tiny_hi.txt \
    --lcpt_output data/raw/tinystories_hi_lcpt.txt \
    --sft_output data/raw/tinystories_translate.jsonl \
    --max_pairs 25000

print("\n✅ outputs:")
!wc -l data/raw/tinystories_hi_lcpt.txt data/raw/tinystories_translate.jsonl

## Step 6 — Validate, dedup, token-check, split (~4 min)

Streams every JSONL row: drops malformed/duplicate/over-length rows, checks token
counts with the actual model tokenizer, writes an 80/10/10 train/val/test split,
and builds the LEAP-format training file.


In [ ]:
# %%time
!python scripts/validate_data.py --num_threads 2 --batch_size 2048

# Convert to LEAP messages format ({"id", "messages": [...]}) — needed for leap-finetune later
!python scripts/convert_to_leap_format.py \
    --input data/validated/sft_train.jsonl \
    --output data/leap/sft_hindi_hinglish_train.jsonl
!python scripts/convert_to_leap_format.py \
    --input data/validated/sft_val.jsonl \
    --output data/leap/sft_val.jsonl

# Step 7 (Phase 1): Continued Pre-Training — LCPT

**Goal:** teach the model to *read* Hindi fluently before teaching it to *chat*.

**Why this matters (the key insight of this project):** the LFM2.5 tokenizer was not
optimized for Devanagari — common Hindi words fragment into 4–17 tokens (English
averages ~4 chars/token; Hindi here ~0.6). The base model has seen almost no Hindi,
so SFT alone would produce broken grammar. LCPT on 60,000 natural Hindi stories
builds the missing fluency.

**Config:** `configs/lcpt_config_t4.yaml` → fp16 (T4 has no bf16), 1 epoch,
60K stories (~1–1.5 h on T4). Loss should drop from ~4 to under 2.5.


In [ ]:
# %%time
# Sanity-check the config first
!cat configs/lcpt_config_t4.yaml

In [ ]:
# %%time
# ~60–90 min on T4. Watch the loss fall — that's the model learning Hindi.
!python scripts/run_lcpt.py --config configs/lcpt_config_t4.yaml

### ✅ LCPT checkpoint quiz (read before continuing)
- The loss went from ~4 → ~2 (if much higher, something's wrong — check the logs)
- `output/lcpt/lcpt_model/` now exists and contains `model.safetensors`
- If Colab disconnected during training: just re-run the cells above — data survives
  (it's on disk), but training restarts from scratch. For long runs consider saving
  checkpoints to Drive.


# Step 8 (Phase 2): Supervised Fine-Tuning (SFT) — teach it to chat

LCPT taught the model to *read* Hindi. SFT now teaches it to *respond*:
it sees formatted conversations (user asks → assistant answers in Hindi/Hinglish)
and learns the response behaviour.

**What happens:**
- LoRA adapters (rank 16, alpha 32) attach to the model — only ~1-2% of weights train
- Loss is computed on the assistant's answer tokens only
- 40,000 curated examples (Hindi + Hinglish + translation pairs), 3 epochs
- fp16 on T4, gradient checkpointing on
- **Expected time on T4: 30-60 minutes**

The config `configs/sft_config_t4.yaml` starts from the LCPT model (`output/lcpt/lcpt_model`).
If you skipped LCPT, first run the next cell to switch it to the base model.


In [ ]:
# If you SKIPPED the LCPT step (Step 6), run this cell so SFT starts from the base model.
# If you ran LCPT, skip this cell.
import yaml, pathlib
p = pathlib.Path("configs/sft_config_t4.yaml")
cfg = yaml.safe_load(p.read_text())
cfg["model_name_or_path"] = "LiquidAI/LFM2.5-350M-Base"
p.write_text(yaml.dump(cfg, allow_unicode=True, sort_keys=False))
print("sft config now starts from:", cfg["model_name_or_path"])


In [ ]:
# %%time
# --- SFT TRAINING (30-60 min on T4) ---
!python scripts/run_sft.py --config configs/sft_config_t4.yaml


## Step 9 — What just happened? Where are the outputs?

After training finishes you will find:

```
output/sft_lora_t4/
├── adapter/     ← LoRA adapter only (~10-20 MB) — needs the base model to run
├── merged/      ← full standalone model (~680 MB) — this is your final model
└── training/    ← checkpoints + logs
```

The `merged/` folder is a complete HuggingFace model you can share, quantize, or run anywhere.

Let's immediately test it — chat with your Hindi model!


In [ ]:
# %%time
# --- TEST YOUR FINE-TUNED MODEL ---
!python scripts/demo.py --model_path output/sft_lora_t4/merged


In [ ]:
# Chat interactively — type your own Hindi / Hinglish prompts
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "output/sft_lora_t4/merged"
tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype=torch.float16, trust_remote_code=True
).cuda()
model.eval()

def chat(prompt, max_new_tokens=200):
    text = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True,
    )
    ids = tok(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=0.7, top_p=0.9)
    return tok.decode(out[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True)

# Try your own prompts here:
for p in [
    "नमस्ते! आप कैसे हैं?",
    "Bhai, mujhe Python seekhna hai, kahan se start karun?",
    "एक छोटी सी कहानी सुनाओ।",
    "What is the capital of India? Answer briefly.",
]:
    print("🧑 USER:", p)
    print("🤖 MODEL:", chat(p))
    print("-" * 60)


## 📦 Step 9 — Export GGUF (run models locally with llama.cpp / Ollama / LM Studio)

GGUF is the file format used by llama.cpp-based tools. We convert the merged model to Q8_0
(near-lossless, ~380MB for 350M). You can change `--quant` to Q4_K_M etc. if you also build
llama.cpp (K-quants need the `llama-quantize` binary from a llama.cpp checkout).

**This step is optional** — skip it if you only want the HF-format model.


In [ ]:
# %%time
# Convert merged HF model -> GGUF (F16/BF16/F32/Q8_0 work out of the box)
!cd leap-finetune && uv run leap-finetune export \
    /content/LFM2.5-350M-Base/output/sft_lora_t4/merged \
    --quant Q8_0

# List the output
!ls -lh /content/LFM2.5-350M-Base/output/sft_lora_t4/merged/gguf/


## 🧪 Step 10 — Test the GGUF with llama.cpp (optional)

We download a prebuilt llama.cpp binary and run a quick Hindi chat with the quantized model.
If the download link fails, grab a release from https://github.com/ggml-org/llama.cpp/releases


In [ ]:
# Download prebuilt llama.cpp (Ubuntu x64 build)
!wget -q https://github.com/ggml-org/llama.cpp/releases/latest/download/llama-bXXXX-bin-ubuntu-x64.zip -O /tmp/llamacpp.zip || echo "download failed - see releases page"
!mkdir -p /tmp/llamacpp && cd /tmp/llamacpp && (unzip -o /tmp/llamacpp.zip 2>/dev/null || echo "no zip")

# Chat test in Hindi (raw prompt format for LFM2.5)
GGUF=$(ls /content/LFM2.5-350M-Base/output/sft_lora_t4/merged/gguf/*.gguf | head -1)
/tmp/llamacpp/llama-cli -m "$GGUF" \
  -p "<|im_start|>user\nनमस्ते! आप कैसे हैं?<|im_end|>\n<|im_start|>assistant\n" \
  -n 128 --temp 0.7 2>/dev/null || echo "llama-cli not available — test GGUF locally in Ollama/LM Studio instead"


## 💾 Step 11 — Save everything to Google Drive (IMPORTANT before session ends!)

Colab deletes ALL files when the session ends. Copy the merged model + GGUF to Drive so
you keep your trained model. The merged model (~680MB) + Q8_0 GGUF (~380MB) fit in free Drive.


In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/lfm25-350m-hindi'
os.makedirs(dest, exist_ok=True)

# 1. Merged HF model (the main artifact)
shutil.copytree('/content/LFM2.5-350M-Base/output/sft_lora_t4/merged',
                f'{dest}/merged', dirs_exist_ok=True)

# 2. GGUF quantized model
gguf_dir = '/content/LFM2.5-350M-Base/output/sft_lora_t4/merged/gguf'
if os.path.isdir(gguf_dir):
    shutil.copytree(gguf_dir, f'{dest}/gguf', dirs_exist_ok=True)

# 3. LoRA adapter (small, useful for future merges)
shutil.copytree('/content/LFM2.5-350M-Base/output/sft_lora_t4/adapter',
                f'{dest}/adapter', dirs_exist_ok=True)

print('✅ Saved to Drive:')
for root, dirs, files in os.walk(dest):
    for fn in files:
        p = os.path.join(root, fn)
        print(f'  {os.path.getsize(p)/1e6:8.1f} MB  {p.replace(dest, "")}')


## 🚀 Step 12 — (Optional) Publish to Hugging Face Hub

Share your Hindi/Hinglish model with the world. Create a write token at
https://huggingface.co/settings/tokens and paste it when prompted.


In [ ]:
from huggingface_hub import login, HfApi

login()  # paste your HF write token

api = HfApi()
api.create_repo('YOUR_USERNAME/LFM2.5-350M-Hindi-Hinglish', exist_ok=True, repo_type='model')
api.upload_folder(
    folder_path='/content/LFM2.5-350M-Base/output/sft_lora_t4/merged',
    repo_id='YOUR_USERNAME/LFM2.5-350M-Hindi-Hinglish',
    repo_type='model',
)
print('🎉 Model published! Check your HF profile.')


# 🎓 What you just learned (finetuning crash course recap)

| Concept | Where you used it |
|---|---|
| **LCPT / continued pretraining** | Step 7 — raw Hindi stories taught the model Hindi *fluency* (next-word prediction) |
| **SFT / instruction tuning** | Step 8 — conversations taught it to *follow instructions* in Hindi/Hinglish |
| **LoRA** | SFT step — trained tiny adapter matrices instead of all 350M weights (fast, small, no forgetting) |
| **Chat template** | Training data was wrapped in `<|im_start|>user...<|im_end|>` markers the model understands |
| **Merging** | LoRA weights were folded back into the base model for a standalone checkpoint |
| **Quantization / GGUF** | Step 9 — compressed to Q8_0 for llama.cpp/Ollama/LM Studio on your own machine |

## 📈 Ideas for improving the model next
1. **More LCPT**: raise `lcpt_max_stories` to 200000 (3-4h on T4) — biggest quality lever for Hindi
2. **More epochs**: try 4-5 SFT epochs, watch val loss for overfitting
3. **Higher LoRA rank**: r=32, alpha=64 in `configs/sft_config_t4.yaml`
4. **Scale up**: same pipeline with `LiquidAI/LFM2.5-1.2B-Base` (needs `lora: true` for LCPT on T4)
5. **DPO**: after SFT, preference-tune with LEAP's DPO support for nicer response style

## 🆘 Troubleshooting
| Problem | Fix |
|---|---|
| `CUDA out of memory` (SFT) | Halve `per_device_train_batch_size` (4→2), double `gradient_accumulation_steps` |
| Session died mid-training | Reconnect, re-run from Step 1; training resumes are not automatic — checkpoints in `output/*/training/checkpoint-*` can be loaded |
| Very slow download of datasets | It's 2.6GB — normal. Grab a coffee ☕ |
| Loss = nan | Ensure `fp16: true` and `bf16: false` in configs (T4 requirement) |
| Model outputs gibberish English | LCPT was too short — increase `lcpt_max_stories` |
| Hindi quality is meh | It's a 350M model — expect simple, correct Hindi; use 1.2B for better quality |
